# analysis.cooccurrence

This notebook shows the coocurrence analyis employing the methodology stated by Veech et al. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import igraph as ig
import networkx as nx
import matplotlib.pyplot as plt
import random
from daforfer import DaforferDB
from scipy import stats
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

## Codetection

We want to illustrate the huge number of codetections that had been obtained by our study. We compute codetection using a similar approach to the generation of the row-organism column-library matrix.


In [ ]:
# bacteria_hits = pd.read_csv("output/hits.bacteria.csv", sep=";").query('is_pab == True')
# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")

bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()

1. We merge virus and bacteria hits by **library**, generating quite a bulky dataframe.
2. We use that dataframe to build a network that we can represent in the article (it should look like a giant hair ball).
3. We save the graph, so we can load it with iGraph.

In [ ]:
coexistence_list = pd.merge(
    virus_hits[['library', 'taxid', 'scientific_name']], 
    bacteria_hits[['library', 'taxid', 'scientific_name', 'pab_type']], 
    on='library', suffixes=['_virus', '_bacteria']
).drop_duplicates(['taxid_bacteria', 'taxid_virus'], keep='first')


In [ ]:
M = nx.Graph()
for _, item in coexistence_list.iterrows():
    M.add_node(item.scientific_name_virus, kingdom='Virus')
    M.add_node(item.scientific_name_bacteria, kingdom='Bacteria')
    M.add_edge(item.scientific_name_virus, item.scientific_name_bacteria)

nx.write_graphml(M, "output/network.coexistance.virusbact-bylibrary.graphml")

Now we represent it.

In [ ]:
node_palette = {
    'host': "#73deac",
    'Virus': "#e58bb1",
    'Bacteria': "#8bc8e5",
}

g = ig.read("output/network.coexistance.virusbact-bylibrary.graphml", format='graphml')
node_colors = [node_palette[node['kingdom']] for node in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]

labels = [str(i) if d > 1000 else "" for i, (d, v) in enumerate(zip(degrees, g.vs))]
random.seed(5)
layout = g.layout("lgl")
ig.plot(
    g, layout=layout, target='figures/network.coexistance-bacteria-virus.svg',
    vertex_color=node_colors, edge_color='black', 
    vertex_size=normalized_sizes, vertex_label=labels, vertex_label_size=10,
    bbox=(400, 400)
)

The coexistance list might be an interesting piece of data for later analysis. We save it in both CSV and Spreadsheet format. 

In [ ]:
db.save_dataframe(
    df=coexistence_list, table_name="D_virusBactCodetect",
    description="Each row of this table is a bacteria and virus detected in the same library"
)

In [ ]:
M.number_of_edges()

## Coocurrence

We load the cooccurrence results, which were obtained through the R library ´cooccur´. The results of this library is a dataframe which contains some entries (not all of them) of the codetection list, together with their probabilities of
cooccurrence

In [ ]:
coocurrence_results = pd.read_csv(
    "output/coocurrence.virusbact-bylibrary.csv", sep=',', index_col=0
)
coocurrence_results

In the following block, we assign kingdom labels to `sp1` and `sp2` in each pair, so we can classify the cooccurrences between `virus-virus`, `bacteria-virus`, or `virus-bacteria`.

In [ ]:
bacteria_names = bacteria_hits['scientific_name'].to_list()
virus_names = virus_hits['scientific_name'].to_list()
name_kingdom_map = {}
for b in bacteria_names:
    name_kingdom_map[b] = 'Bacteria'
for v in virus_names:
    name_kingdom_map[v] = 'Virus'
coocurrence_results['sp1_kingdom'] = coocurrence_results['sp1_name'].map(name_kingdom_map)
coocurrence_results['sp2_kingdom'] = coocurrence_results['sp2_name'].map(name_kingdom_map)
coocurrence_results = coocurrence_results.dropna(subset=['sp1_name', 'sp2_name'])

coocurrence_results

The following tables counts the number of positive coocurrences between kingdoms.

In [ ]:
negative_cooccurrences = coocurrence_results.query('p_lt < 0.05').value_counts(['sp1_kingdom', 'sp2_kingdom']).reset_index()
positive_cooccurrences = coocurrence_results.query('p_gt < 0.05').value_counts(['sp1_kingdom', 'sp2_kingdom']).reset_index()
negative_cooccurrences['type'] = negative_cooccurrences.apply(lambda x: "-".join(sorted((x.sp1_kingdom, x.sp2_kingdom))), axis=1)
positive_cooccurrences['type'] = positive_cooccurrences.apply(lambda x: "-".join(sorted((x.sp1_kingdom, x.sp2_kingdom))), axis=1)
coocurrences_by_type = pd.merge(positive_cooccurrences[['type', 'count']], negative_cooccurrences[['type', 'count']], on='type', suffixes=['_positive', '_negative'], how='outer').fillna(0)
coocurrences_by_type['count_negative'] = coocurrences_by_type['count_negative'].astype(int)
coocurrences_by_type['count_positive'] = coocurrences_by_type['count_positive'].astype(int)

db.save_dataframe(coocurrences_by_type, "D_coocByType", description="Number and type of cooccurrences between bacteria/virus")
coocurrences_by_type

### Full coocurrence network

In [ ]:
coocurrence_results = coocurrence_results.query('p_gt < 0.05 or p_lt < 0.05')
db.save_dataframe(
    df=coocurrence_results, table_name="D_cooccAll",
    description="Significative results (0.05) from cooccurrence analysis, including bacteria-bacteria and virus-virus cooccurrences"
)


In [ ]:
M = nx.Graph()
for _, item in coocurrence_results.query('p_gt < 0.05').iterrows():
    M.add_edge(item.sp1_name, item.sp2_name, sign="positive")
for _, item in coocurrence_results.query('p_lt < 0.05').iterrows():
    M.add_edge(item.sp1_name, item.sp2_name, sign="negative")

nx.write_graphml(M, "output/network.coocurrence.virusbact-bylibrary.graphml")

In [ ]:

for node, node_attributes in M.nodes(data=True):
    node_attributes['kingdom'] = name_kingdom_map[node]
    node_attributes['name'] = node

nodes_df = pd.DataFrame.from_records([node_attributes for _, node_attributes in M.nodes(data=True)])
nodes_df

In [ ]:
nodes_df.value_counts(['kingdom'])

In [ ]:
sign_palette = {
    'positive': "#b23044",
    'negative': "#4d5db9",
    
}
node_palette = {
    'host': "#73deac",
    'Virus': "#e58bb1",
    'Bacteria': "#8bc8e5",
}
name_kingdom_map['nan'] = 'Virus'
g = ig.read("output/network.coocurrence.virusbact-bylibrary.graphml", format='graphml')
edge_colors = [sign_palette[edge['sign']] for edge in g.es]
node_colors = [node_palette[name_kingdom_map[node['id']]] for node in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]

labels = [str(i) if d > 0 else "" for i, (d, v) in enumerate(zip(degrees, g.vs))]
random.seed(124)
layout = g.layout("lgl")
ig.plot(
    g, layout=layout, target='figures/network.coocurrence-bacteria-virus.svg',
    vertex_color=node_colors, edge_color=edge_colors, 

    vertex_size=normalized_sizes, vertex_label=labels, vertex_label_size=10,
    bbox=(400, 400)
)

### Virus-bacteria only network

To build the bacteria-virus only network, we keep only those edges where the kingdoms are different.

In [ ]:
coocurrence_results = coocurrence_results.query('p_gt < 0.05 or p_lt < 0.05')
coocurrence_results = coocurrence_results.query('sp1_kingdom != sp2_kingdom')

db.save_dataframe(
    df=coocurrence_results, table_name="D_cooccBactVirus",
    description="Significative results (0.05) from cooccurrence analysis, including only virus-bacteria cooccurrences"
)
coocurrence_results

In [ ]:
M = nx.MultiGraph()
for _, item in coocurrence_results.query('p_gt < 0.05').iterrows():
    if item.sp1_kingdom != item.sp2_kingdom:
        M.add_edge(item.sp1_name, item.sp2_name, sign="positive")
for _, item in coocurrence_results.query('p_lt < 0.05').iterrows():
    if item.sp1_kingdom != item.sp2_kingdom:
        M.add_edge(item.sp1_name, item.sp2_name, sign="negative")

nx.write_graphml(M, "output/network.coocurrence.virusbact-bylibrary.trans.graphml")

In [ ]:

for node, node_attributes in M.nodes(data=True):
    node_attributes['kingdom'] = name_kingdom_map[node]
    node_attributes['name'] = node

nodes_df = pd.DataFrame.from_records([node_attributes for _, node_attributes in M.nodes(data=True)])
nodes_df.value_counts(['kingdom']) # q000023 q000024

In [ ]:
name_kingdom_map['nan'] = 'Virus'
g = ig.read("output/network.coocurrence.virusbact-bylibrary.trans.graphml", format='graphml')
edge_colors = [sign_palette[edge['sign']] for edge in g.es]
node_colors = [node_palette[name_kingdom_map[node['id']]] for node in g.vs]
degrees = g.degree()
min_degree = min(degrees)
max_degree = max(degrees)
degree_threshold = 2
normalized_sizes = [10 + 20 * (degree - min_degree) / (max_degree - min_degree) for degree in degrees]
labels = [str(i) if d >= degree_threshold else "" for i, (d, v) in enumerate(zip(degrees, g.vs))]
random.seed(12)
layout = g.layout("lgl")
ig.plot(
    g, layout=layout, target='figures/network.coocurrence-bacteria-virus.trans.svg',
    vertex_color=node_colors, edge_color=edge_colors, 

    vertex_size=normalized_sizes, vertex_label=labels, vertex_label_size=10,
    bbox=(300, 300)
)

In [ ]:
assert(M.number_of_edges() == 57)
assert(M.number_of_nodes() == 38)

In [ ]:
network_node_labels = []
for i, (d, v) in enumerate(zip(degrees, g.vs)):
    if d >= degree_threshold:
        network_node_labels.append({"item": i, "degree": d, "name": v["id"]})
network_node_labels = pd.DataFrame.from_records(network_node_labels)

db.save_dataframe(
    df=network_node_labels, table_name="P_coocNetworkLabels",
    description="Labels of the coocccurence network figure, with their degree"
)


In [ ]:


coocurrence_results['coocurrence_type'] = coocurrence_results.apply(lambda x: "-".join(sorted([x.sp1_kingdom, x.sp2_kingdom])), axis=1)
coocurrence_results

### Cooccurrence as a heatmap 

To be 1000% confident that the cooccurrence matrix that we are displaying is not the result of a code error, we will represent the heatmap from the original data.

In [ ]:
coocurrence_results = coocurrence_results.copy()
coocurrence_results['positive'] = coocurrence_results['p_gt'] < 0.05
coocurrence_results['negative'] = coocurrence_results['p_lt'] < 0.05
coocurrence_results['cooccurrence_type'] = coocurrence_results.apply(lambda x: x.positive - x.negative, axis=1)
coocurrence_results['P'] = coocurrence_results.apply(lambda x: x.p_gt if x.cooccurrence_type == 1 else x.p_lt, axis=1)
coocurrence_results['P'] = coocurrence_results.apply(lambda x: - x.cooccurrence_type * np.log10(x.P + 1e-8) if x.cooccurrence_type != 0 else 0.0, axis=1)
coocurrence_results_pvt =  coocurrence_results.query('cooccurrence_type != 0').query('coocurrence_type == "Bacteria-Virus"').pivot(
    index='sp1_name', columns='sp2_name', values='P'
).fillna(0)

In [ ]:

sns.clustermap(coocurrence_results_pvt, cmap='bwr', square=True, figsize=(7.5, 6), row_cluster=False, col_cluster=False, cbar_pos=(0.05, 0.45, 0.025, 0.40), linecolor = 'gray', linewidth=0.5, vmin=-3.0, vmax=3.0)

In [ ]:
db.save_dataframe(
    df=coocurrence_results_pvt, table_name="P_coocHeatMap",
    description="Cooccurrence matrix, weighting each positive or negative interaction by the log-p-value"
)

### Virus-bacteria cooccurrence network properties

We use the same kind of calculation that we will use later for the properties of bipartite host-virus/bacteria networks. 

#### Write down adjacency matrix

In [ ]:
coocurrence_results['foo'] = (coocurrence_results['cooccurrence_type'] != 0.0).astype(int)
coocurrence_results_pvt =  coocurrence_results.query('cooccurrence_type != 0').query('coocurrence_type == "Bacteria-Virus"').pivot(
    index='sp1_name', columns='sp2_name', values='foo'
).fillna(0)

coocurrence_results_pvt.to_csv("scratch/adjmat.cooccurrence-virus-bacteria.csv", sep=";")
coocurrence_results_pvt

#### Process bipartite results

This includes a null file and and observation file.

In [ ]:
null = pd.read_csv("scratch/bipartite.cooccurrence-virus-bacteria.null.csv")
observed = pd.read_csv("scratch/bipartite.cooccurrence-virus-bacteria.observed.csv")
observed

#### Plots

Connectance plot

In [ ]:
g = sns.displot(data=null, x='connectance', kind='kde', aspect=1.0, fill=True, height=2.0, color='gray')
m : float = float(observed.set_index('Metric').loc['connectance', 'Value']) # pyright: ignore[reportArgumentType]
g.ax.axvline(m, linestyle='--', color='red')

In [ ]:
g = sns.displot(data=null, x='NODF', kind='kde', aspect=1.0, fill=True, height=2.0, color='gray')
g.ax.axvline(observed.set_index('Metric').loc['NODF', 'Value'], linestyle='--', color='red') # pyright: ignore[reportArgumentType]

In [ ]:
g = sns.displot(data=null, x='modularity Q', kind='kde', aspect=1.0, fill=True, height=2.0, color='gray')
g.ax.axvline(observed.set_index('Metric').loc['modularity Q'].Value, linestyle='--', color='red') # pyright: ignore[reportArgumentType]

#### Compute Z-scores

We compute the Z-score against the null model substracting the mean of the null model and dividing the standard deviation.

In [ ]:
Z_scores = pd.merge(
    pd.merge(
        null.reset_index().melt(id_vars=['index'], value_vars=['modularity Q', 'connectance', 'NODF']).groupby('variable', as_index=False)['value'].mean().rename(columns={'value': 'null_mean'}), # pyright: ignore[reportCallIssue]
        null.reset_index().melt(id_vars=['index'], value_vars=['modularity Q', 'connectance', 'NODF']).groupby('variable', as_index=False)['value'].std().rename(columns={'value': 'null_std'}),  # pyright: ignore[reportCallIssue]
        on='variable'
    ),
    observed, left_on='variable', right_on='Metric'
)
Z_scores['Z-score'] = (Z_scores['Value'] - Z_scores['null_mean']) / Z_scores['null_std']
Z_scores

#### Compute t-test

We use the 1-sample t-test below to compute the p-value of our observation being the mean of the null model.

In [ ]:
ttest = []
for metric in ['NODF', 'connectance', 'modularity Q']:
    v = observed.set_index('Metric').loc[metric].Value
    t_stat, p_val = stats.ttest_1samp(null[metric].values, v)
    ttest.append({"Metric": metric, "p-value": p_val, "ttest-stat": t_stat})

ttest_df = pd.DataFrame.from_records(ttest)
ttest_df

### Save

In [ ]:
network_statistics = pd.merge(Z_scores, ttest_df, on='Metric')
db.save_dataframe(
    df=network_statistics, table_name="T_coocBipStats",
    description="Cooccurrence network connectance, Modularity and Nestedness (NODF) analyzed using null model r2d-derived Z-scores and t-test 1 sample"
)

### Cooccurrence network analysis using log(P-value) weights

In [ ]:
# coocurrence_results['foo'] = (coocurrence_results['cooccurrence_type'] != 0.0).astype(int)
# coocurrence_results['P_abs'] = coocurrence_results['P'].abs()
# coocurrence_results_pvt =  coocurrence_results.query('cooccurrence_type != 0').query('coocurrence_type == "Bacteria-Virus"').pivot(
#     index='sp1_name', columns='sp2_name', values='P_abs'
# ).fillna(0)

# coocurrence_results_pvt.to_csv("scratch/adjmat.cooccurrence-virus-bacteria.abs.csv", sep=";")
# coocurrence_results_pvt

In [ ]:
# null = pd.read_csv("scratch/bipartite.cooccurrence-virus-bacteria.abs.null.csv")
# observed = pd.read_csv("scratch/bipartite.cooccurrence-virus-bacteria.abs.observed.csv")
# Z_scores = pd.merge(
#     pd.merge(
#         null.reset_index().melt(id_vars='index', value_vars=['modularity Q', 'connectance', 'NODF']).groupby('variable', as_index=False)['value'].mean().rename(columns={'value':'null_mean'}),# pyright: ignore[reportCallIssue]
#         null.reset_index().melt(id_vars='index', value_vars=['modularity Q', 'connectance', 'NODF']).groupby('variable', as_index=False)['value'].std().rename(columns={'value':'null_std'}), # pyright: ignore[reportCallIssue]
#         on='variable'
#     ),
#     observed, left_on='variable', right_on='Metric'
# )
# Z_scores['Z-score'] = (Z_scores['Value'] - Z_scores['null_mean']) / Z_scores['null_std']
# Z_scores

In [ ]:
db.conn.close()